In [1]:
# %%
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import polars as pl
import pandas as pd
import numpy as np
import json
import os
import sys

sys.path.append("..")

import seaborn as sns

sns.set()
from settings import (
    random_state,
    PROJECT_PATH,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)
from catboost import CatBoostRegressor



In [2]:
transactions = pl.read_parquet(
    os.path.join(PROJECT_PATH, "real_estate_transactions_engineered.parquet")
)

In [3]:
X = transactions.drop([REGRESSION_TARGET, CLASSIFICATION_TARGET]).to_pandas()
y_regression = transactions[REGRESSION_TARGET].to_pandas()

In [4]:
with open("../features_used.json", "r") as f:
    feature_names = json.load(f)

with open("../categorical_features_used.json", "r") as f:
    categorical_features = json.load(f)

numerical_features = [col for col in feature_names if col not in categorical_features]

In [5]:
catboost_regressor = CatBoostRegressor(random_state=random_state, verbose=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_regression, random_state=random_state
)

In [8]:
catboost_regressor.fit(X_train[feature_names], y_train)

In [9]:
y_pred_train = catboost_regressor.predict(X_train)
errors = y_train - y_pred_train    
X_train["error"] = errors

y_pred_test = catboost_regressor.predict(X_test)
errors = y_test - y_pred_test    
X_test["error"] = errors

The most classic chart for visualising the relationship between a categorical feature and a quantitative one is the boxplot. However, our visualisation library seaborn cannot handle the data as-is with the One Hot Encoding we applied.

We first need to reverse this step in order to obtain a raw categorical column that we can pass to seaborn for plotting.


In [10]:
# Looking at our features, we need to reverse the encoding for region and building type 
categorical_features

['building_type_apartment',
 'building_type_house',
 'off_plan',
 'city_requested',
 'region_auvergne_rhone_alpes',
 'region_nouvelle_aquitaine',
 'region_occitanie',
 "region_provence_alpes_cote_dazur",
 'region_ile_de_france']

In [11]:
def reverse_one_hot_encoding(X, feature_name, ):

    X[feature_name] = pd.from_dummies(
        X[[col for col in X.columns if col.startswith(feature_name)]]
    )

    X = X.drop(
        [col for col in X.columns if col.startswith(feature_name + "_")], axis=1
    )

    return X

In [12]:
# We create "group" dataframes to avoid modifying the original features
X_train_group = reverse_one_hot_encoding(X_train, "region_name")
X_test_group = reverse_one_hot_encoding(X_test, "region_name")

X_train_group = reverse_one_hot_encoding(X_train_group, "building_type")
X_test_group = reverse_one_hot_encoding(X_test, "building_type")

In [13]:
feature_names_updated = [col for col in feature_names if col in X_train_group.columns]
feature_names_updated.extend(["region_name", "building_type"])

In [14]:
categorical_features_updated = [col for col in feature_names_updated if col not in numerical_features]

In [20]:
# Our categorical columns have been restored to their raw state! 
categorical_features_updated

['off_plan', 'city_requested', 'region_name', 'building_type']

In [21]:
def plot_error_violinplot(X, categorical_features):

    for feature in categorical_features:
        plt.figure()
        sns.boxplot(x=X[feature], y=X["error"])
        plt.ylabel("Error")
        plt.xlabel(feature)
        plt.xticks(rotation=70)
        plt.title("Boxplot of Error vs {feature}")
        plt.show()


This analysis reveals that we are dealing with several atypical values and even outliers, which tend to concentrate around specific values of certain features, for example:
* Buildings without a VEFA plan
* Cities that are not under tension (city_requested = 0)
* The Île-de-France region


In [ ]:
plot_error_violinplot(X_test_group, categorical_features_updated)